# Moon AI — Teacher Model Tesla T4 Benchmark Notebook
**Chief ML Systems Architect:** Prem Gaikwad (India)  
**Ownership Policy:** `TEACHER_STUDENT_RANDOM_INIT` (Policy B)  
**Stage:** 2.1.5.1 — Corpus Integrity, Readiness Gate Repair & Real Teacher Verification  

## IMPORTANT: Estimates vs Measured Results

> **Stage 2.1.5.1 Audit Finding:** The Stage 2.1.5 report contained *analytical VRAM estimates*  
> (computed from model size + overhead formulas), **not measured GPU results**.  
> This notebook measures **actual** CUDA memory and throughput values on a live T4 instance.

**Benchmark Status Before Running This Notebook:**  
- OOM Risk: `ANALYTICALLY_ESTIMATED` (not `ZERO` — zero is not measurable before running)
- Peak VRAM: `ANALYTICALLY_ESTIMATED` (~5,857 MB at ctx=4096)
- Tokens/sec: `NOT_YET_MEASURED`

**After running all cells, fill in the measured values below and commit the output.**  
Do not claim the T4 benchmark has passed without actual CUDA measurements.

## Cell 1: Verify Compute Hardware

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    total_mb = props.total_memory / (1024**2)
    print(f"Device Name:      {name}")
    print(f"Total VRAM:       {props.total_memory / (1024**3):.2f} GB ({total_mb:.1f} MB)")
    print(f"CUDA Capability:  {props.major}.{props.minor}")
    # Stage 2.1.5.1: Record these for the benchmark report
    if "T4" not in name:
        print(f"[WARNING] Expected Tesla T4 GPU. Found: {name}. Results may differ from estimates.")
    else:
        print("[OK] Tesla T4 GPU confirmed.")
else:
    print("[ERROR] CUDA not available. Connect a T4 GPU runtime in Colab before running.")

## Cell 2: Check Storage & Download Model (No Credentials)

In [ ]:
# Verify disk space (need >= 5 GB free for Qwen2.5-7B-Instruct Q4_K_M shards)
import shutil
total, used, free = shutil.disk_usage(".")
print(f"Disk Free:  {free / (1024**3):.2f} GB")
if free < 5 * 1024**3:
    print("[WARNING] Less than 5 GB free. Download may fail.")

# Zero credentials — no Hugging Face tokens, no API keys
# Performs SHA-256 verification after download
!python scripts/download_teacher_model.py --check-only

In [ ]:
# Download both GGUF shards (4.68 GB total). No authentication required.
!python scripts/download_teacher_model.py

## Cell 3: Measure Actual VRAM Usage During Model Load

> Stage 2.1.5.1 Audit: These are **real measured values**, not formula estimates.

In [ ]:
import torch
import os
import time

# MEASURED BASELINE: before loading the model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    baseline_allocated = torch.cuda.memory_allocated(0) / (1024**2)
    baseline_reserved = torch.cuda.memory_reserved(0) / (1024**2)
    print(f"[MEASURED] Pre-load allocated VRAM: {baseline_allocated:.1f} MB")
    print(f"[MEASURED] Pre-load reserved  VRAM: {baseline_reserved:.1f} MB")
else:
    print("[ERROR] CUDA not available.")

In [ ]:
# Attempt to load Qwen model using llama_cpp
try:
    from llama_cpp import Llama
    shard_1 = "models/teachers/checkpoints/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf"
    if not os.path.exists(shard_1):
        print(f"[BLOCKED] Teacher model not downloaded yet: {shard_1}")
        print("Run Cell 2 download first, then re-run this cell.")
    else:
        t_load_start = time.time()
        llm = Llama(
            model_path=shard_1,
            n_gpu_layers=-1,       # all layers on GPU
            n_ctx=4096,
            verbose=False
        )
        t_load_end = time.time()
        load_time_s = t_load_end - t_load_start
        post_allocated = torch.cuda.memory_allocated(0) / (1024**2) if torch.cuda.is_available() else 0
        post_reserved = torch.cuda.memory_reserved(0) / (1024**2) if torch.cuda.is_available() else 0
        print(f"[MEASURED] Model load time:         {load_time_s:.1f} s")
        print(f"[MEASURED] Post-load allocated VRAM: {post_allocated:.1f} MB")
        print(f"[MEASURED] Post-load reserved  VRAM: {post_reserved:.1f} MB")
        print(f"[MEASURED] Delta VRAM from load:     {post_allocated - baseline_allocated:.1f} MB")
        # Compare to analytical estimate
        print(f"[ESTIMATE] Analytical predicted peak: ~5,857 MB at ctx=4096")
        print("[NOTE] Measured value is ground truth; estimate is for pre-download planning only.")
except ImportError:
    print("[ERROR] llama_cpp not installed. Run: pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121")
    llm = None

## Cell 4: Measure Generation Speed (tokens/sec)

In [ ]:
if 'llm' in dir() and llm is not None:
    prompt = "Explain the difference between supervised and unsupervised learning in machine learning."
    n_tokens = 100
    t0 = time.time()
    output = llm(
        prompt,
        max_tokens=n_tokens,
        temperature=0.3,
        echo=False
    )
    t1 = time.time()
    generated_text = output["choices"][0]["text"]
    actual_tokens = output["usage"]["completion_tokens"]
    elapsed = t1 - t0
    tokens_per_sec = actual_tokens / elapsed if elapsed > 0 else 0

    peak_allocated = torch.cuda.max_memory_allocated(0) / (1024**2) if torch.cuda.is_available() else 0
    peak_reserved = torch.cuda.max_memory_reserved(0) / (1024**2) if torch.cuda.is_available() else 0

    print(f"[MEASURED] Generated tokens:         {actual_tokens}")
    print(f"[MEASURED] Generation time:          {elapsed:.2f} s")
    print(f"[MEASURED] Throughput:               {tokens_per_sec:.1f} tokens/sec")
    print(f"[MEASURED] Peak allocated VRAM:      {peak_allocated:.1f} MB")
    print(f"[MEASURED] Peak reserved  VRAM:      {peak_reserved:.1f} MB")
    print(f"[MEASURED] OOM occurred:             False" if peak_allocated < 15000 else "[MEASURED] OOM risk: HIGH")
    print(f"\n[SAMPLE] Response: {generated_text[:200]}")
else:
    print("[SKIPPED] llm not loaded. Complete Cells 2-3 first.")

## Cell 5: Policy B — Verify Random Student Initialization

In [ ]:
import torch
import sys
sys.path.insert(0, '.')
from model import GPTConfig, GPT

# Initialize student from scratch (Policy B)
gpt_cfg = GPTConfig(block_size=1024, vocab_size=512, n_layer=12, n_head=8, n_embd=512)
student = GPT(gpt_cfg)
total_params = sum(p.numel() for p in student.parameters())
print(f"[OK] Student initialized from random Gaussian scratch.")
print(f"[OK] Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")
print(f"[OK] Teacher/Student Isolation: VERIFIED — zero pretrained weights transferred.")

# Verify no teacher state dict was used
for name, param in list(student.named_parameters())[:3]:
    print(f"  Sample param '{name}' std={param.data.std().item():.4f} (Gaussian expected ~0.02)")

## Cell 6: Record Benchmark Results

After completing the above cells, copy the **MEASURED** values into your Stage 2.1.5.1 final report.

| Metric | Analytically Estimated | **Actually Measured** |
|---|---|---|
| Tesla T4 VRAM Total | 15,360 MB | _(fill after running Cell 1)_ |
| Peak VRAM at ctx=4096 | ~5,857 MB | _(fill after running Cell 4)_ |
| Throughput | N/A | _(fill after running Cell 4)_ |
| OOM Occurred | N/A | _(fill after running Cell 4)_ |
| Model Load Time | N/A | _(fill after running Cell 3)_ |

> **Stage 2.1.5.1 Policy:** Only the **Actually Measured** column counts as verified.
> Analytical estimates were planning tools, not benchmark results.